# Stimulus-Dependent Functional Network Topology in Mouse Visual Cortex (Extended)

**Topic:** Network Neuroscience Applied to Visual Processing Across Six Visual Areas

**Overview**

This tutorial demonstrates how to analyze functional connectivity in neural populations using graph theory methods. We investigate whether different visual stimuli (natural images vs. Gabor patches) induce distinct network topologies across **six areas** of mouse visual cortex (V1, LM, RL, AL, PM, AM), using data from the Allen Brain Observatory.

This is the extended version of the original BSAI Final Project. The original analyzed only V1 and LM; this version generalizes the pipeline to all six visual cortex areas recorded by Neuropixels probes.

**Learning Objectives:**
1. Load and preprocess Neuropixels data from the Allen Brain Observatory across multiple brain areas
2. Compute within-area and cross-area functional connectivity using Pearson correlation
3. Apply graph theory metrics (modularity, clustering coefficient) to each area
4. Perform multi-session statistical analysis with effect sizes across the visual hierarchy
5. Interpret stimulus-dependent network differences across the cortical visual hierarchy

**Connection to Course (Lecture 11: Network Neuroscience)**
- `NetworkX` for graph construction and analysis
- `python-louvain` for community detection (modularity)
- Correlation-based functional connectivity
- Cohen's d effect size interpretation

---

In [1]:
# Install required packages
# Option 1: Using pip (recommended)
# pip install -r requirements.txt

# Option 2: Using conda
'''
conda create -n neuroai python=3.11 -y
conda activate neuroai
pip install -r requirements.txt
'''

# Note: First run will download ~50GB from Allen Institute

'\nconda create -n neuroai python=3.11 -y\nconda activate neuroai\npip install -r requirements.txt\n'

## Part 1: Setup and Data Loading

We use the Allen Brain Observatory Visual Behavior Neuropixels dataset, which provides high-density extracellular recordings from mouse visual cortex.

**Six Visual Cortex Areas:**
- **V1 (VISp)**: Primary visual cortex - initial processing of visual features
- **LM (VISl)**: Lateromedial area - higher-order visual processing
- **RL (VISrl)**: Rostrolateral area - motion and spatial processing
- **AL (VISal)**: Anterolateral area - object and pattern processing
- **PM (VISpm)**: Posteromedial area - spatial navigation and scene processing
- **AM (VISam)**: Anteromedial area - visuomotor integration

**Analysis Parameters:**
- Bin size: 50 ms
- Correlation threshold: 0.1
- Minimum firing rate: 0.1 Hz
- Minimum neurons per area: 5
- Random seed: 42

In [2]:
# Cell 1: Setup and imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
from scipy import stats as scipy_stats

# Add src to path
sys.path.insert(0, os.path.abspath('./src'))

from data_loading import (
    VISUAL_AREAS, AREA_NAMES,
    load_cache, get_session_list, load_session,
    get_units_with_areas, get_area_neurons, get_spike_times,
    get_stimulus_presentations, get_stimulus_times,
    compute_firing_rates_for_stimulus,
    load_session_data, compute_firing_rates_by_area
)
from connectivity import (
    compute_all_connectivity_matrices,
    summarize_connectivity
)
from graph_metrics import (
    compute_all_metrics,
    compute_cross_area_metrics,
    summarize_metrics
)

# For reproducibility
np.random.seed(42)

# Analysis parameters
BIN_SIZE = 0.050    # 50 ms bins
THRESHOLD = 0.1     # Correlation threshold for adjacency
MIN_RATE = 0.1      # Hz - minimum firing rate to include neuron
MIN_NEURONS = 5     # Minimum good neurons for an area to be included

# Create output directories
os.makedirs('./results/figures', exist_ok=True)
os.makedirs('./results/multi_session', exist_ok=True)

print("[OK] Imports successful")
print(f"\nVisual areas: {[f'{AREA_NAMES[a]} ({a})' for a in VISUAL_AREAS]}")
print(f"\nAnalysis parameters:")
print(f"  Bin size: {BIN_SIZE*1000:.0f} ms")
print(f"  Correlation threshold: {THRESHOLD}")
print(f"  Minimum firing rate: {MIN_RATE} Hz")
print(f"  Minimum neurons per area: {MIN_NEURONS}")

[OK] Imports successful

Visual areas: ['V1 (VISp)', 'LM (VISl)', 'RL (VISrl)', 'AL (VISal)', 'PM (VISpm)', 'AM (VISam)']

Analysis parameters:
  Bin size: 50 ms
  Correlation threshold: 0.1
  Minimum firing rate: 0.1 Hz
  Minimum neurons per area: 5


In [3]:
# Cell 2: Load Allen Brain Observatory cache
# NOTE: First run will download ~50GB of data from Allen Institute servers

CACHE_DIR = './data/allen_cache/'
cache = load_cache(CACHE_DIR)
sessions = get_session_list(cache)

print(f"[OK] Cache loaded")
print(f"Total sessions available: {len(sessions)}")

ecephys_sessions.csv: 100%|███████████████████████████████████████████| 64.4k/64.4k [00:00<00:00, 222kMB/s]
behavior_sessions.csv: 100%|███████████████████████████████████████████| 562k/562k [00:00<00:00, 1.16MMB/s]
units.csv: 100%|███████████████████████████████████████████████████████| 134M/134M [00:08<00:00, 15.6MMB/s]
probes.csv: 100%|███████████████████████████████████████████████████████| 130k/130k [00:00<00:00, 343kMB/s]
channels.csv: 100%|██████████████████████████████████████████████████| 27.9M/27.9M [00:02<00:00, 13.3MMB/s]


[OK] Cache loaded
Total sessions available: 103


## Part 2: Session Selection and Per-Area Coverage

We use the same 65 sessions from the original project (documented in `session_ids.txt`). These sessions were originally selected to have sufficient V1 and LM neurons, but Neuropixels probes record from all six visual areas simultaneously. Not every session has neurons in all six areas -- the pipeline gracefully handles variable area coverage by analyzing whichever areas have at least `MIN_NEURONS` good units per session.

In [4]:
# Cell 3: Scan per-area neuron coverage across all sessions
unit_table = cache.get_unit_table()

print(f"Scanning all sessions for neuron counts in {len(VISUAL_AREAS)} visual areas...")
print(f"Minimum neurons per area: {MIN_NEURONS}\n")

good_sessions = []
for session_id in tqdm(sessions.index, desc="Scanning sessions"):
    sess_units = unit_table[unit_table['ecephys_session_id'] == session_id]
    if 'quality' in sess_units.columns:
        sess_units = sess_units[sess_units['quality'] == 'good']
    
    row = {'session_id': session_id}
    for area in VISUAL_AREAS:
        row[f'n_{area}'] = int((sess_units['structure_acronym'] == area).sum())
    row['n_areas_present'] = sum(1 for a in VISUAL_AREAS if row[f'n_{a}'] >= MIN_NEURONS)
    good_sessions.append(row)

good_sessions_df = pd.DataFrame(good_sessions)

print(f"\n[OK] Scanned {len(good_sessions_df)} sessions")
print(f"\nPer-area coverage (sessions with >= {MIN_NEURONS} neurons):")
for area in VISUAL_AREAS:
    col = f'n_{area}'
    n_with = (good_sessions_df[col] >= MIN_NEURONS).sum()
    mean_n = good_sessions_df.loc[good_sessions_df[col] >= MIN_NEURONS, col].mean()
    print(f"  {AREA_NAMES[area]:>2s} ({area}): {n_with}/{len(good_sessions_df)} sessions"
          f"  (mean {mean_n:.0f} neurons when present)" if n_with > 0 else
          f"  {AREA_NAMES[area]:>2s} ({area}): {n_with}/{len(good_sessions_df)} sessions")

Scanning all sessions for neuron counts in 6 visual areas...
Minimum neurons per area: 5



Scanning sessions: 100%|████████████████████████████████████████████████| 103/103 [00:00<00:00, 679.16it/s]


[OK] Scanned 103 sessions

Per-area coverage (sessions with >= 5 neurons):
  V1 (VISp): 101/103 sessions  (mean 120 neurons when present)
  LM (VISl): 98/103 sessions  (mean 111 neurons when present)
  RL (VISrl): 99/103 sessions  (mean 89 neurons when present)
  AL (VISal): 101/103 sessions  (mean 99 neurons when present)
  PM (VISpm): 102/103 sessions  (mean 121 neurons when present)
  AM (VISam): 99/103 sessions  (mean 102 neurons when present)


## Part 3: Single-Session Analysis Demo

We demonstrate the full analysis pipeline on a single session before scaling to all 65 sessions. The pipeline now loads all six visual areas (where present), computes within-area and between-area connectivity, and extracts graph metrics for each area.

### 3.1 Data Loading and Firing Rate Computation

In [5]:
# Cell 4: Load a single session for demonstration (all 6 areas)
example_session_id = good_sessions_df.iloc[0]['session_id']

data = load_session_data(
    cache_dir=CACHE_DIR,
    session_id=int(example_session_id),
    quality_filter=True,
    min_neurons=MIN_NEURONS
)

print(f"\n[OK] Session {data['session_id']} loaded")
print(f"  Areas present ({len(data['areas_present'])}): "
      f"{[AREA_NAMES[a] for a in data['areas_present']]}")
print(f"  Neuron counts: { {AREA_NAMES[a]: n for a, n in data['neuron_counts'].items()} }")

Loading cache...
Loading session 1044385384...


ecephys_session_1044385384.nwb: 100%|████████████████████████████████| 2.65G/2.65G [03:22<00:00, 13.1MMB/s]
/Users/donespejo/Documents/TsinghuaFall2025/Machine Learning/BSAI_Final_Project_Extended/venv/lib/python3.11/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Getting units with area info...
  V1 (VISp): 56 neurons
  LM (VISl): 154 neurons
  RL (VISrl): 104 neurons
  AL (VISal): 75 neurons
  PM (VISpm): 132 neurons
  AM (VISam): 0 neurons [below threshold of 5]
Getting stimulus presentations...
Areas with >= 5 neurons: 5/6

[OK] Session 1044385384 loaded
  Areas present (5): ['V1', 'LM', 'RL', 'AL', 'PM']
  Neuron counts: {'V1': 56, 'LM': 154, 'RL': 104, 'AL': 75, 'PM': 132, 'AM': 0}


In [6]:
# Cell 5: Get stimulus times (spike times already loaded in data['spike_times_by_area'])
stim = data['stim_presentations']
natural_starts, natural_ends = get_stimulus_times(stim, 'natural')
gabor_starts, gabor_ends = get_stimulus_times(stim, 'gabor')

print(f"Natural image presentations: {len(natural_starts)}")
print(f"  Total duration: {(natural_ends - natural_starts).sum():.1f}s")
print(f"\nGabor presentations: {len(gabor_starts)}")
print(f"  Total duration: {(gabor_ends - gabor_starts).sum():.1f}s")
print(f"\nSpike times loaded for {len(data['spike_times_by_area'])} areas")

Natural image presentations: 9600
  Total duration: 2402.5s

Gabor presentations: 3645
  Total duration: 912.0s

Spike times loaded for 5 areas


In [ ]:
# Cell 6: Compute firing rates for all areas
print("Computing firing rates for all areas...")

rates_natural = compute_firing_rates_by_area(
    data['spike_times_by_area'], natural_starts, natural_ends, bin_size=BIN_SIZE
)
rates_gabor = compute_firing_rates_by_area(
    data['spike_times_by_area'], gabor_starts, gabor_ends, bin_size=BIN_SIZE
)

print(f"\n[OK] Firing rates computed for {len(rates_natural)} areas")
for area in rates_natural:
    fr, uids = rates_natural[area]
    print(f"  {AREA_NAMES.get(area, area)} ({area}): {fr.shape[0]} neurons x {fr.shape[1]} bins")

Computing firing rates for all areas...


### 3.2 Functional Connectivity

Functional connectivity is defined as the **Pearson correlation coefficient** between neuron pairs:

$$\rho_{ij} = \frac{\text{cov}(r_i, r_j)}{\sigma_{r_i} \sigma_{r_j}}$$

where $r_i$ and $r_j$ are the firing rate time series of neurons $i$ and $j$.

In [ ]:
# Cell 7: Compute connectivity matrices for all areas
print("Computing connectivity matrices...")

fr_natural = {area: rates for area, (rates, _) in rates_natural.items()}
uid_natural = {area: uids for area, (_, uids) in rates_natural.items()}

fr_gabor = {area: rates for area, (rates, _) in rates_gabor.items()}
uid_gabor = {area: uids for area, (_, uids) in rates_gabor.items()}

matrices_natural = compute_all_connectivity_matrices(
    fr_natural, uid_natural, min_rate_threshold=MIN_RATE
)
matrices_gabor = compute_all_connectivity_matrices(
    fr_gabor, uid_gabor, min_rate_threshold=MIN_RATE
)

print(f"\n[OK] Connectivity computed")
print(f"\nWithin-area matrices ({len(matrices_natural['within'])}):")
for area, (corr, mask) in matrices_natural['within'].items():
    print(f"  {AREA_NAMES.get(area, area)} ({area}): {corr.shape}")

print(f"\nBetween-area matrices ({len(matrices_natural['between'])}):")
for (a1, a2), (cross, m1, m2) in matrices_natural['between'].items():
    print(f"  {AREA_NAMES.get(a1, a1)}-{AREA_NAMES.get(a2, a2)}: {cross.shape}")

In [ ]:
# Cell 8: Visualize within-area connectivity matrices (all areas)
within_areas = list(matrices_natural['within'].keys())
n_areas = len(within_areas)

fig, axes = plt.subplots(n_areas, 2, figsize=(10, 4 * n_areas))
if n_areas == 1:
    axes = axes.reshape(1, -1)

for row, area in enumerate(within_areas):
    corr_nat = matrices_natural['within'][area][0]
    corr_gab = matrices_gabor['within'][area][0] if area in matrices_gabor['within'] else None
    name = AREA_NAMES.get(area, area)

    axes[row, 0].imshow(corr_nat, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    axes[row, 0].set_title(f'{name} ({area}) - Natural')
    axes[row, 0].set_xlabel('Neuron')
    axes[row, 0].set_ylabel('Neuron')

    if corr_gab is not None:
        im = axes[row, 1].imshow(corr_gab, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
        axes[row, 1].set_title(f'{name} ({area}) - Gabor')
    else:
        axes[row, 1].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[row, 1].transAxes)
        axes[row, 1].set_title(f'{name} ({area}) - Gabor')
        im = axes[row, 0].images[0]
    axes[row, 1].set_xlabel('Neuron')

fig.colorbar(im, ax=axes, label='Correlation', shrink=0.4, pad=0.02)
plt.suptitle('Within-Area Functional Connectivity Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/figures/connectivity_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("[OK] Figure saved to results/figures/connectivity_matrices.png")

### 3.3 Graph Construction and Network Metrics

We convert correlation matrices to graphs using thresholding:

$$A_{ij} = \begin{cases} 1 & \text{if } |\rho_{ij}| \geq \theta \\ 0 & \text{otherwise} \end{cases}$$

where $\theta = 0.1$ is the correlation threshold.

**Key Network Metrics:**
- **Modularity (Q)**: Measures how well a network divides into communities. Computed using the Louvain algorithm (Blondel et al., 2008).
- **Clustering Coefficient**: Measures local connectivity density.
- **Density**: Ratio of actual edges to possible edges.

In [ ]:
# Cell 9: Compute graph metrics for all areas
print("Network Metrics (Single Session)")
print("=" * 70)

demo_metrics = {}
for area in within_areas:
    corr_nat = matrices_natural['within'][area][0]
    nat_m = compute_all_metrics(corr_nat, threshold=THRESHOLD)
    
    if area in matrices_gabor['within']:
        corr_gab = matrices_gabor['within'][area][0]
        gab_m = compute_all_metrics(corr_gab, threshold=THRESHOLD)
    else:
        gab_m = None
    
    demo_metrics[area] = (nat_m, gab_m)
    
    name = AREA_NAMES.get(area, area)
    print(f"\n{name} ({area}):")
    print(f"  {'Metric':<25} {'Natural':<15} {'Gabor':<15}")
    print(f"  {'-'*55}")
    print(f"  {'Modularity Q':<25} {nat_m['modularity']['modularity_Q']:<15.4f} "
          f"{gab_m['modularity']['modularity_Q']:<15.4f}" if gab_m else
          f"  {'Modularity Q':<25} {nat_m['modularity']['modularity_Q']:<15.4f} {'N/A':<15}")
    print(f"  {'Clustering (mean)':<25} {nat_m['clustering']['mean']:<15.4f} "
          f"{gab_m['clustering']['mean']:<15.4f}" if gab_m else
          f"  {'Clustering (mean)':<25} {nat_m['clustering']['mean']:<15.4f} {'N/A':<15}")
    print(f"  {'Density':<25} {nat_m['density']:<15.4f} "
          f"{gab_m['density']:<15.4f}" if gab_m else
          f"  {'Density':<25} {nat_m['density']:<15.4f} {'N/A':<15}")
    print(f"  {'# Communities':<25} {nat_m['modularity']['n_communities']:<15d} "
          f"{gab_m['modularity']['n_communities']:<15d}" if gab_m else
          f"  {'# Communities':<25} {nat_m['modularity']['n_communities']:<15d} {'N/A':<15}")

## Part 4: Multi-Session Analysis (All 6 Areas)

We now scale the analysis to **N = 65 sessions** across all six visual areas to obtain statistically robust results.

For each session, within-area connectivity and graph metrics are computed for every area with sufficient neurons. Results are stored with per-area columns so that areas not present in a given session are simply NaN.

The sessions used are listed in `session_ids.txt` for reproducibility.

In [ ]:
# Cell 10: Load session IDs from file
with open('./session_ids.txt', 'r') as f:
    lines = f.readlines()

# Parse session IDs (skip comment lines)
valid_session_ids = [int(line.strip()) for line in lines if line.strip() and not line.startswith('#')]
print(f"Loaded {len(valid_session_ids)} valid session IDs from session_ids.txt")

In [ ]:
# Cell 11: Process all sessions across all 6 areas (with checkpointing)
# NOTE: This cell takes ~30-60 minutes on first run
# Results are checkpointed -- if interrupted, re-run to resume
#
# IMPORTANT: If you previously ran the old V1/LM-only version, delete the
# old checkpoint file first since the column format has changed:
#   rm ./results/multi_session/checkpoint_results_6area.csv

CHECKPOINT_FILE = './results/multi_session/checkpoint_results_6area.csv'

if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    completed_sessions = set(existing_df['session_id'].values)
    all_results = existing_df.to_dict('records')
    print(f"[RESUME] Resuming: {len(completed_sessions)} sessions already processed")
else:
    completed_sessions = set()
    all_results = []
    print("[NEW] Starting fresh")

sessions_to_process = [s for s in valid_session_ids if s not in completed_sessions]
print(f"[...] {len(sessions_to_process)} sessions remaining to process\n")

for session_id in tqdm(sessions_to_process, desc="Processing sessions"):
    try:
        sess_data = load_session_data(
            cache_dir=CACHE_DIR, session_id=session_id,
            quality_filter=True, min_neurons=MIN_NEURONS
        )
        
        if len(sess_data['areas_present']) == 0:
            continue
        
        stim = sess_data['stim_presentations']
        nat_starts, nat_ends = get_stimulus_times(stim, 'natural')
        gab_starts, gab_ends = get_stimulus_times(stim, 'gabor')
        
        if len(nat_starts) == 0 or len(gab_starts) == 0:
            continue
        
        rates_nat = compute_firing_rates_by_area(
            sess_data['spike_times_by_area'], nat_starts, nat_ends, bin_size=BIN_SIZE
        )
        rates_gab = compute_firing_rates_by_area(
            sess_data['spike_times_by_area'], gab_starts, gab_ends, bin_size=BIN_SIZE
        )
        
        fr_nat = {a: r for a, (r, _) in rates_nat.items()}
        uid_nat = {a: u for a, (_, u) in rates_nat.items()}
        fr_gab = {a: r for a, (r, _) in rates_gab.items()}
        uid_gab = {a: u for a, (_, u) in rates_gab.items()}
        
        mat_nat = compute_all_connectivity_matrices(fr_nat, uid_nat, min_rate_threshold=MIN_RATE)
        mat_gab = compute_all_connectivity_matrices(fr_gab, uid_gab, min_rate_threshold=MIN_RATE)
        
        conn_nat = summarize_connectivity(mat_nat)
        conn_gab = summarize_connectivity(mat_gab)
        
        result = {
            'session_id': session_id,
            'n_natural_stim': len(nat_starts),
            'n_gabor_stim': len(gab_starts),
        }
        
        for area in VISUAL_AREAS:
            result[f'n_{area}'] = sess_data['neuron_counts'].get(area, 0)
            
            if area in mat_nat['within']:
                m_nat = compute_all_metrics(mat_nat['within'][area][0], threshold=THRESHOLD)
                result[f'{area}_modularity_natural'] = m_nat['modularity']['modularity_Q']
                result[f'{area}_clustering_natural'] = m_nat['clustering']['mean']
                result[f'{area}_density_natural'] = m_nat['density']
                key_nat = f'within_{area}'
                if key_nat in conn_nat:
                    result[f'{area}_corr_natural'] = conn_nat[key_nat]['mean']
            
            if area in mat_gab['within']:
                m_gab = compute_all_metrics(mat_gab['within'][area][0], threshold=THRESHOLD)
                result[f'{area}_modularity_gabor'] = m_gab['modularity']['modularity_Q']
                result[f'{area}_clustering_gabor'] = m_gab['clustering']['mean']
                result[f'{area}_density_gabor'] = m_gab['density']
                key_gab = f'within_{area}'
                if key_gab in conn_gab:
                    result[f'{area}_corr_gabor'] = conn_gab[key_gab]['mean']
        
        all_results.append(result)
        completed_sessions.add(session_id)
        pd.DataFrame(all_results).to_csv(CHECKPOINT_FILE, index=False)
        
    except Exception as e:
        print(f"\nSession {session_id} failed: {str(e)[:80]}")
        continue

results_df = pd.DataFrame(all_results)
print(f"\n[OK] Processed {len(results_df)} sessions successfully")
print(f"\nPer-area data availability:")
for area in VISUAL_AREAS:
    col = f'{area}_modularity_natural'
    if col in results_df.columns:
        n_valid = results_df[col].notna().sum()
        print(f"  {AREA_NAMES[area]} ({area}): {n_valid}/{len(results_df)} sessions")
    else:
        print(f"  {AREA_NAMES[area]} ({area}): 0/{len(results_df)} sessions")

## Part 5: Statistical Analysis

We use **paired t-tests** since each session provides matched Natural vs. Gabor comparisons.

**Effect Size (Cohen's d for paired samples):**

$$d = \frac{\bar{X}_{diff}}{s_{diff}}$$

where $\bar{X}_{diff}$ is the mean difference and $s_{diff}$ is the standard deviation of differences.

**Interpretation (Cohen, 1988):**
- |d| < 0.2: negligible
- 0.2 ≤ |d| < 0.5: small
- 0.5 ≤ |d| < 0.8: medium
- |d| ≥ 0.8: large

In [ ]:
# Cell 12: Statistical analysis across all areas

def cohens_d_paired(condition1, condition2):
    """Cohen's d for paired samples."""
    diff = condition1 - condition2
    return np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 0.0

def interpret_effect(d):
    d_abs = abs(d)
    if d_abs < 0.2: return "negligible"
    elif d_abs < 0.5: return "small"
    elif d_abs < 0.8: return "medium"
    else: return "large"

valid_df = results_df[results_df['n_natural_stim'] > 1000].copy()
n_total = len(valid_df)
print(f"Statistical Analysis (N = {n_total} sessions total)")
print("=" * 80)

MIN_SESSIONS_FOR_STATS = 10
metrics_to_test = ['modularity', 'corr']
stat_results = {}

n_tests = 0
for area in VISUAL_AREAS:
    for metric in metrics_to_test:
        col_nat = f'{area}_{metric}_natural'
        col_gab = f'{area}_{metric}_gabor'
        if col_nat not in valid_df.columns or col_gab not in valid_df.columns:
            continue
        paired = valid_df[[col_nat, col_gab]].dropna()
        if len(paired) >= MIN_SESSIONS_FOR_STATS:
            n_tests += 1

alpha_corrected = 0.05 / n_tests if n_tests > 0 else 0.05
print(f"Bonferroni correction: {n_tests} tests, corrected alpha = {alpha_corrected:.4f}\n")

all_significant = True
for area in VISUAL_AREAS:
    name = AREA_NAMES[area]
    area_has_data = False
    
    for metric in metrics_to_test:
        col_nat = f'{area}_{metric}_natural'
        col_gab = f'{area}_{metric}_gabor'
        if col_nat not in valid_df.columns or col_gab not in valid_df.columns:
            continue
        
        paired = valid_df[[col_nat, col_gab]].dropna()
        n_paired = len(paired)
        if n_paired < MIN_SESSIONS_FOR_STATS:
            continue
        
        if not area_has_data:
            print(f"\n{name} ({area}) -- N = {n_paired} sessions:")
            area_has_data = True
        
        vals_nat = paired[col_nat].values
        vals_gab = paired[col_gab].values
        t_stat, p_val = scipy_stats.ttest_rel(vals_nat, vals_gab)
        d_val = cohens_d_paired(vals_nat, vals_gab)
        sig = "***" if p_val < alpha_corrected else "n.s."
        
        metric_label = "Modularity" if metric == "modularity" else "Correlation"
        print(f"  {metric_label}:")
        print(f"    Natural: {np.mean(vals_nat):.4f} +/- {np.std(vals_nat):.4f}")
        print(f"    Gabor:   {np.mean(vals_gab):.4f} +/- {np.std(vals_gab):.4f}")
        print(f"    t({n_paired-1}) = {t_stat:.2f}, p = {p_val:.2e}, "
              f"d = {d_val:.2f} ({interpret_effect(d_val)}) {sig}")
        
        stat_results[(area, metric)] = {
            'n': n_paired, 't': t_stat, 'p': p_val, 'd': d_val,
            'nat_mean': np.mean(vals_nat), 'nat_std': np.std(vals_nat),
            'gab_mean': np.mean(vals_gab), 'gab_std': np.std(vals_gab),
            'significant': p_val < alpha_corrected,
        }
        if p_val >= alpha_corrected:
            all_significant = False

print(f"\n{'=' * 80}")
print(f"Bonferroni-corrected alpha = {alpha_corrected:.4f} ({n_tests} tests)")
print(f"All findings significant after correction: {'YES' if all_significant else 'NO'}")

## Part 6: Results Visualization

In [ ]:
# Cell 13: Main results figure -- all areas with sufficient data
plot_areas = [a for a in VISUAL_AREAS if (a, 'modularity') in stat_results]
n_plot = len(plot_areas)

if n_plot == 0:
    print("No areas with sufficient data for plotting.")
else:
    fig, axes = plt.subplots(n_plot, 2, figsize=(12, 4 * n_plot))
    if n_plot == 1:
        axes = axes.reshape(1, -1)

    for row, area in enumerate(plot_areas):
        name = AREA_NAMES[area]
        
        for col, metric in enumerate(['modularity', 'corr']):
            ax = axes[row, col]
            sr = stat_results.get((area, metric))
            if sr is None:
                ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12)
                ax.set_title(f'{name} {"Modularity" if metric == "modularity" else "Correlation"}')
                continue
            
            col_nat = f'{area}_{metric}_natural'
            col_gab = f'{area}_{metric}_gabor'
            paired = valid_df[[col_nat, col_gab]].dropna()
            vals_nat = paired[col_nat].values
            vals_gab = paired[col_gab].values
            n_p = len(paired)
            
            for i in range(n_p):
                ax.plot([0, 1], [vals_nat[i], vals_gab[i]], 'o-',
                        color='gray', alpha=0.2, linewidth=0.5, markersize=3)
            
            ax.errorbar([0], [sr['nat_mean']], yerr=[sr['nat_std'] / np.sqrt(n_p)],
                        fmt='s', markersize=12, color='#2E7D32', capsize=6, linewidth=2.5)
            ax.errorbar([1], [sr['gab_mean']], yerr=[sr['gab_std'] / np.sqrt(n_p)],
                        fmt='s', markersize=12, color='#E65100', capsize=6, linewidth=2.5)
            
            ax.set_xlim(-0.5, 1.5)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Natural', 'Gabor'], fontsize=11)
            metric_label = 'Modularity Q' if metric == 'modularity' else 'Mean Correlation'
            ax.set_ylabel(f'{name} {metric_label}', fontsize=11)
            sig_marker = '***' if sr['significant'] else 'n.s.'
            ax.set_title(f'{name} {metric_label}\np = {sr["p"]:.2e}, d = {sr["d"]:.2f} {sig_marker}',
                         fontsize=12, fontweight='bold')
            ax.grid(alpha=0.3)

    plt.suptitle(f'Multi-Session Results Across Visual Hierarchy (N = {n_total})',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('./results/figures/main_results_6area.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("[OK] Figure saved to results/figures/main_results_6area.png")

## Part 7: Interpretation and Conclusions

### Key Findings

The results above show how stimulus-dependent network topology varies across the visual cortical hierarchy. The original project established that natural images induce higher modularity than Gabor patches in V1 and LM. This extended analysis reveals whether that pattern holds, weakens, or reverses in higher visual areas (RL, AL, PM, AM).

Key questions answered by the 6-area extension:
1. **Is the modularity effect consistent across the hierarchy?** Do all areas show Natural > Gabor modularity, or does it vary?
2. **Does the correlation pattern hold?** The original project found Gabor > Natural correlation in V1/LM.
3. **Are there hierarchical gradients?** Do effect sizes systematically change from lower (V1) to higher (AM) areas?

### Mechanistic Interpretation

**Why Natural Images Tend Toward Higher Modularity:**
- Natural images contain diverse features (edges, textures, objects)
- Different neuron subpopulations specialize for different features
- This creates distinct functional modules

**Why Gabor Patches Tend Toward Lower Modularity:**
- Gabor patches are "optimal stimuli" for orientation-tuned V1 neurons (Hubel & Wiesel, 1962)
- Many neurons respond similarly, producing uniform activation
- No distinct functional groups emerge

**Cross-Hierarchy Considerations:**
- Higher areas (AL, PM, AM) receive more processed, mixed inputs
- Their responses to simple stimuli like Gabors may differ qualitatively from V1
- The presence or absence of the modularity effect in higher areas informs whether this is a local or global property of visual processing

### Limitations

1. Secondary data analysis (Allen preprocessing pipeline)
2. Mouse visual cortex may not generalize to primates/humans
3. Passive viewing paradigm (no behavioral task)
4. Single correlation threshold (though robustness was tested)
5. Not all sessions have neurons in all 6 areas (variable coverage)
6. Pearson correlation captures undirected, unsigned connectivity only

### Future Directions (Dimensions 2 and 3)

1. **Directed Connectivity (CCGs)**: Replace Pearson correlation with jitter-corrected cross-correlograms to obtain directed, signed (excitatory/inhibitory) connectivity matrices
2. **Directed Graph Analysis**: Use nx.DiGraph for motif analysis (16 three-neuron patterns), multi-regional module detection, and hierarchical signal flow analysis
3. **Machine Learning Classification**: Use per-area modularity/clustering as features to classify stimulus type

In [ ]:
# Cell 14: Summary table across all areas
print("\n" + "=" * 100)
print(f"SUMMARY TABLE: Multi-Session Results (N = {n_total} sessions, {n_tests} tests)")
print("=" * 100)
print(f"{'Area':<6} {'Metric':<14} {'N':<5} {'Natural (mean +/- SEM)':<24} "
      f"{'Gabor (mean +/- SEM)':<24} {'d':<10} {'p-value':<12} {'Sig?':<5}")
print("-" * 100)

for area in VISUAL_AREAS:
    name = AREA_NAMES[area]
    for metric in ['modularity', 'corr']:
        sr = stat_results.get((area, metric))
        if sr is None:
            continue
        n_p = sr['n']
        nat_sem = sr['nat_std'] / np.sqrt(n_p)
        gab_sem = sr['gab_std'] / np.sqrt(n_p)
        metric_label = 'Modularity' if metric == 'modularity' else 'Correlation'
        sig_str = 'YES' if sr['significant'] else 'no'
        print(f"{name:<6} {metric_label:<14} {n_p:<5} "
              f"{sr['nat_mean']:.4f} +/- {nat_sem:.4f}          "
              f"{sr['gab_mean']:.4f} +/- {gab_sem:.4f}          "
              f"{sr['d']:<10.2f} {sr['p']:<12.2e} {sig_str:<5}")

print("-" * 100)
print(f"Bonferroni-corrected alpha = {alpha_corrected:.4f} ({n_tests} tests)")
print(f"\n[OK] Tutorial complete!")

---

## References

1. Siegle, J.H., Jia, X., et al. (2021). Survey of spiking in the mouse visual system reveals functional hierarchy. *Nature*, 592, 86-92.

2. Tang, D., Zylberberg, J., Jia, X., & Choi, H. (2024). Stimulus type shapes the topology of cellular functional networks in mouse visual cortex. *Nature Communications*, 15, 5753.

3. Jia, X., et al. (2022). Multi-regional module-based signal transmission in mouse visual cortex. *Neuron*, 110(8), 1328-1343.

4. Rubinov, M., & Sporns, O. (2010). Complex network measures of brain connectivity: uses and interpretations. *NeuroImage*, 52(3), 1059-1069.

5. Blondel, V.D., et al. (2008). Fast unfolding of communities in large networks. *Journal of Statistical Mechanics*, P10008.

6. Bassett, D.S., & Sporns, O. (2017). Network neuroscience. *Nature Neuroscience*, 20(3), 353-364.

7. Cohen, J. (1988). Statistical Power Analysis for the Behavioral Sciences. Lawrence Erlbaum Associates.

---

**Course:** Brain Science and Artificial Intelligence (Dr. Xiaoxuan Jia)  
**Institution:** Tsinghua University  
**Date:** December 2025 (original) / February 2026 (extended)